# Gaussian Naive Bayes

This notebook is one standalone implementation for the ML capstone Review 1. Run cells top-to-bottom. All notebooks use `random_state=42` and the same 80:20 split for fair comparison.

## 1. Imports

In [ ]:
# Libraries for data handling, preprocessing, Naive Bayes, and evaluation.
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

from sklearn.naive_bayes import GaussianNB

## 2. Dataset audit, target definition and preprocessing

In [ ]:
# Load, clean, and prepare the Car Dekho classification data.
# Dataset and classification target
DATA_URL = "https://raw.githubusercontent.com/rohithtej2109/Smart-car-deals/b438f32fc37ae91e6f8a72ef912705b2c48b427f/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv"
TARGET = "transmission"  # Chosen binary target: Manual vs Automatic.

df = pd.read_csv(DATA_URL)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Clean target and remove duplicates.
df[TARGET] = df[TARGET].astype(str).str.strip()
df = df.drop_duplicates().dropna(subset=[TARGET]).copy()

# Feature engineering: car_age captures vehicle age.
if "year" in df.columns:
    df["car_age"] = pd.Timestamp.now().year - pd.to_numeric(df["year"], errors="coerce")
    df["car_age"] = df["car_age"].clip(lower=0)

# Avoid high-cardinality name column as a raw categorical feature.
if "name" in df.columns:
    df = df.drop(columns=["name"])

# Keep only the two most common classes if an unexpected rare class exists.
# For this dataset, transmission is normally Manual/Automatic.
valid_classes = df[TARGET].value_counts().index.tolist()
if len(valid_classes) > 2:
    valid_classes = valid_classes[:2]
    df = df[df[TARGET].isin(valid_classes)].copy()

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

print("Shape:", df.shape)
print("Class distribution:")
display(y.value_counts())


## 3. Required EDA

In [ ]:
# EDA checks data quality, class balance, and numerical relationships before modeling.
display(df.head())
display(df.describe(include="all").T)
print("Missing values:\n", df.isna().sum())
plt.figure(figsize=(6,4)); sns.countplot(x=y); plt.title("Class distribution: transmission"); plt.tight_layout(); plt.show()
plt.figure(figsize=(10,6)); sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", center=0); plt.title("Numeric correlation heatmap"); plt.tight_layout(); plt.show()
for col in [c for c in ["year","km_driven","car_age"] if c in df.columns]:
    plt.figure(figsize=(7,4)); sns.boxplot(data=df, x=TARGET, y=col); plt.title(f"{col} by transmission"); plt.tight_layout(); plt.show()


## 4. Model training and tuning

In [ ]:
# Gaussian Naive Bayes estimates class probabilities from the prepared features.
base_model = GaussianNB()
model = Pipeline([("preprocessor", preprocessor), ("model", base_model)])


In [ ]:
# Fit the complete preprocessing + classifier pipeline using only the training data.
model.fit(X_train, y_train)
print('Model fitted.')

## 5. Evaluation

In [ ]:
# Predict unseen test samples and compare the predictions with the true transmission labels.
pred = model.predict(X_test)
proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
average = "weighted"
result = {
    "Model": 'Gaussian Naive Bayes',
    "Accuracy": accuracy_score(y_test, pred),
    "Precision_weighted": precision_score(y_test, pred, average=average, zero_division=0),
    "Recall_weighted": recall_score(y_test, pred, average=average, zero_division=0),
    "F1_weighted": f1_score(y_test, pred, average=average, zero_division=0)
}
if proba is not None:
    if len(np.unique(y_test)) == 2:
        result["ROC_AUC"] = roc_auc_score((y_test == sorted(y_test.unique())[1]).astype(int), proba[:,1])
    else:
        result["ROC_AUC"] = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
display(pd.DataFrame([result]))


## 6. Confusion matrix and model interpretation

In [ ]:
# Use the confusion matrix to inspect the model's classification errors by class.
cm = confusion_matrix(y_test, pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap="Blues"); plt.title("Confusion matrix"); plt.tight_layout(); plt.show()

if hasattr(model.named_steps["model"], "feature_importances_"):
    names = model.named_steps["preprocessor"].get_feature_names_out()
    imp = pd.Series(model.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,5)); imp.sort_values().plot(kind="barh")
    plt.title("Top feature importances"); plt.tight_layout(); plt.show()


## 7.Interpretation

**Observation:** This notebook uses **Gaussian Naive Bayes** to predict the `transmission` class (Manual or Automatic) from the available car features.

- The data is cleaned before modeling, and `car_age` is created as an additional feature.
- Numerical and categorical features are preprocessed separately so that the classifier receives usable numerical input.
- The model is trained on 80% of the data and evaluated on the remaining 20% unseen test data.
- Where GridSearchCV is used, the selected hyperparameters are chosen using 5-fold cross-validation with weighted F1 as the scoring measure.
- The final prediction is evaluated using accuracy, weighted precision, weighted recall, weighted F1, ROC-AUC, and a confusion matrix.
- **Model-specific observation:** Gaussian Naive Bayes uses class probabilities under its feature-independence and Gaussian-distribution assumptions.

**Final observation:** Discuss the actual metric values and confusion matrix produced when the notebook is run. No numerical performance value is assumed here unless it is displayed by the notebook.